In [3]:
import os
import numpy as np
from tqdm import tqdm

from skimage.io import imread
from skimage.transform import resize
from skimage.feature import hog

import torch
import torch.nn as nn
from torchvision import models, transforms


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load ResNet50 (without the classification head)
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
resnet.fc = nn.Identity()       # Remove the final fully connected layer.
resnet = resnet.to(device)
resnet.eval()                    # Set model to evaluation mode

# Preprocessing pipeline aligned with ImageNet requirements
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet standard deviation
    )
])



In [5]:

# We obtain the output of the penultimate layer,
# which consists of 2,048 values, resulting in a 2,048-dimensional feature vector.
@torch.no_grad()
def extract_resnet_features(img):
    img_t = preprocess(img).unsqueeze(0).to(device)     # Apply preprocessing and add batch dimension
    features = resnet(img_t)                            # Forward pass through ResNet (without the FC layer)
    return features.squeeze().cpu().numpy()             # Remove batch dimension, move to CPU, convert to NumPy


In [6]:
from skimage.color import rgb2gray

def extract_hog_features(img):
    img_gray = rgb2gray(img)
    img_gray = resize(img_gray, (128, 64), preserve_range=True)

    features = hog(
        img_gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )
    return features

In [7]:
def load_dataset(path):
    X = []
    y = []

    classes = sorted(os.listdir(path))

    for label, cls in enumerate(classes):
        folder = os.path.join(path, cls)
        
        for file in tqdm(os.listdir(folder), desc=f"Loading {cls}"):
            img_path = os.path.join(folder, file)

            img = imread(img_path)

            # ResNet Features
            f_res = extract_resnet_features(img)

            # HOG Features
            f_hog = extract_hog_features(img)

            # Combine
            f_combined = np.concatenate([f_res, f_hog])

            X.append(f_combined)
            y.append(label)

    return np.array(X), np.array(y)


In [8]:
import os
BASE_PATH = os.path.abspath(".")

TRAIN_DIR = os.path.join(BASE_PATH, "seg_train")
TEST_DIR  = os.path.join(BASE_PATH, "seg_test")

X_train, y_train = load_dataset(TRAIN_DIR)
X_test, y_test = load_dataset(TEST_DIR)

# 2048 (ResNet) + ~3780 (HOG) ≈ 5828 Features each image
# ResNet handles color information 
# HOG handles edge and texture information
print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


Loading street: 100%|██████████| 501/501 [00:09<00:00, 55.11it/s]


Train Shape: (14034, 5828)
Test Shape: (3000, 5828)


In [ ]:
# The linear SVM is particularly well suited for this task, as it separates data effectively 
# in high-dimensional feature spaces and requires significantly less training time and RAM 
# compared to models such as XGBoost.
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import time 
svm = SVC(kernel="linear", probability=True)

start_train = time.time()
svm.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60 

y_pred = svm.predict(X_test)


print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec")

In [ ]:
from sklearn.svm import SVC

# RBF-SVM Modell
svm_rbf = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    probability=False
)

# Training
start_train = time.time()
svm_rbf.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60 

# prediction
pred = svm_rbf.predict(X_test)




print("RBF-SVM Accuracy:", accuracy_score(y_test, pred))
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec")

RBF-SVM Accuracy: 0.9293333333333333
Training Time: 66.13s
Prediction Time: 43.39s


In [ ]:
# k-NN achieves good accuracy in this dataset because the HOG+CNN features form
# well-structured clusters.
# However, the method remains unstable in high-dimensional spaces and can easily
# degrade ("collapse") if the feature distribution changes.
from sklearn.neighbors import KNeighborsClassifier

# kNN-Modell definieren
knn = KNeighborsClassifier(
    n_neighbors=5,     # typische Wahl; kannst du anpassen
    weights='uniform', # oder 'distance'
    metric='minkowski' # Standard, entspricht euklidischer Distanz bei p=2
)

# Training
start_train = time.time()
knn.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60 

# Prediction
pred_knn = knn.predict(X_test)

# Ausgabe
print("kNN Accuracy:", accuracy_score(y_test, pred_knn))
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec")



kNN Accuracy: 0.9203333333333333
Training Time: 0 min 0.09 sec


In [ ]:
# When more CPU cores are used, Random Forest builds more trees in parallel,
# which speeds up training. Since each tree requires its own memory,
# parallel tree construction increases RAM usage while reducing training time.

# However, for high-dimensional feature spaces, Random Forest often needs a larger
# number of trees and deeper trees to effectively learn meaningful splits across
# the entire feature set. This increases model capacity, but also RAM usage and
# training time.
from sklearn.ensemble import RandomForestClassifier


rf = RandomForestClassifier(
    n_estimators=300,       # 300 trees
    max_depth=100,          # Limits tree depth
    n_jobs=-1,              # Utilizes all available CPU cores for parallel tree training
    bootstrap=True          # More stable results and reduced overfitting
)

# Training
start_train = time.time()
rf.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60 

# Prediction
pred_rf = rf.predict(X_test)


print("Random Forest Accuracy:", accuracy_score(y_test, pred_rf))
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec")


Random Forest Accuracy: 0.9173333333333333
Training Time: 49.04s
Prediction Time: 0.18s


In [ ]:
# Gradient Boosting requires significantly more training time because each tree
# is built sequentially — a new tree cannot be trained until the previous one
# is completed. This prevents parallel tree construction and limits CPU
# utilization, making the algorithm much slower than Random Forest.

# In our high-dimensional setting (5828 features), each tree needs to evaluate
# a very large number of potential feature splits. With 200 trees and depth 8,
# the computational workload grows rapidly, leading to long training times.

# Additionally, every tree stores its own parameters (split thresholds, leaf
# values), so the memory footprint increases with each boosting iteration.
# This combination of sequential training, high-dimensional feature evaluation,
# and growing memory requirements explains the long overall training time.

from xgboost import XGBClassifier
import time

# XGBoost model configuration
# These parameters are RAM-friendly and well suited for many features (5828)
# combined with many samples (14k)
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,          # Uses 80% of the training data per tree → reduces overfitting and RAM usage
    colsample_bytree=0.6,   # Uses 60% of the features per tree → reduces memory and computation
    tree_method="hist",     # Extremely RAM- and CPU-efficient → essential for large datasets
    predictor="cpu_predictor"  # CPU mode (GPU would be faster, but is not required here)
)

# Memory reduction: converting float64 → float32 cuts RAM usage in half
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

# Training the XGBoost model
# XGBClassifier handles 14k samples × 5828 features efficiently using histogram-based splitting.
# Without these memory-optimized parameters, XGBoost would be untrainable on this dataset.
start_train = time.time()
xgb.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60

# Prediction
pred = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, pred))
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec")


c:\Users\Med Amin\Downloads\Project2_Intel\Project2_Intel\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:00:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.924
Training Time: 950.0531s
Prediction Time: 0.0839s
